# ForecastCF Baseline sur Colab

Ce notebook lance ForecastCF avec iTransformer sur ETTh1 pour comparer avec votre méthode RL.

**Durée estimée**: 30-60 minutes avec GPU T4

## 1. Setup - Cloner le repo et installer les dépendances

In [ ]:
# Cloner votre repo (remplacez par votre URL GitHub)
!git clone https://github.com/VOTRE_USERNAME/counterfactual-forecasting-rl.git
%cd counterfactual-forecasting-rl

In [ ]:
# Installer les dépendances
!pip install -q torch numpy pandas matplotlib seaborn tensorflow

## 2. Vérifier que tout est en place

In [ ]:
import os
import torch
import tensorflow as tf

print(f"PyTorch version: {torch.__version__}")
print(f"TensorFlow version: {tf.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# Vérifier les fichiers
print("\nFichiers nécessaires:")
files_to_check = [
    "assets/checkpoints/etth1_chpts/forecaster/chpt_etth1_96_48_S.pth",
    "assets/configs/models/etth1_dataset/forecasters/itransformer/etth1_96_48_S.json",
    "assets/datasets/ETTh1.csv",
    "baselines/ForecastCF/src/cf_search_pytorch.py"
]

for f in files_to_check:
    exists = "✓" if os.path.exists(f) else "✗"
    print(f"{exists} {f}")

## 3. Lancer ForecastCF pour tous les seeds

In [ ]:
import subprocess
import time

# Paramètres
model_path = "assets/checkpoints/etth1_chpts/forecaster/chpt_etth1_96_48_S.pth"
config_path = "assets/configs/models/etth1_dataset/forecasters/itransformer/etth1_96_48_S.json"
data_path = "assets/datasets/ETTh1.csv"
output_file = "baselines/ForecastCF/results/forecastcf_etth1_itransformer.csv"

seeds = [1, 9, 30, 33, 39]
device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Device: {device}")
print(f"Seeds: {seeds}")
print(f"Output: {output_file}\n")

total_start = time.time()

for i, seed in enumerate(seeds, 1):
    print(f"\n{'='*80}")
    print(f"[{i}/{len(seeds)}] Seed {seed}")
    print(f"{'='*80}")
    
    start = time.time()
    
    cmd = [
        "python", "baselines/ForecastCF/src/cf_search_pytorch.py",
        "--model-path", model_path,
        "--model-type", "itransformer",
        "--config-path", config_path,
        "--dataset", "etth1",
        "--data-path", data_path,
        "--horizon", "48",
        "--back-horizon", "96",
        "--center", "median",
        "--desired-shift", "0",
        "--desired-change", "-0.1",
        "--poly-order", "1",
        "--fraction-std", "1.0",
        "--random-seed", str(seed),
        "--output", output_file,
        "--device", device,
        "--test-samples", "1000"
    ]
    
    try:
        result = subprocess.run(cmd, check=True, capture_output=True, text=True)
        elapsed = time.time() - start
        print(f"\n✓ Seed {seed} terminé en {elapsed:.1f}s")
        
        # Afficher les dernières lignes de sortie
        if result.stdout:
            lines = result.stdout.strip().split('\n')
            for line in lines[-10:]:
                if 'Validity' in line or 'Proximity' in line or 'Compactness' in line:
                    print(f"  {line}")
    
    except subprocess.CalledProcessError as e:
        print(f"\n✗ Erreur pour seed {seed}")
        if e.stderr:
            print(f"Erreur: {e.stderr[:500]}")
        break

total_elapsed = time.time() - total_start
print(f"\n{'='*80}")
print(f"✓ Tous les seeds terminés en {total_elapsed/60:.1f} minutes")
print(f"{'='*80}")

## 4. Vérifier les résultats

In [ ]:
import pandas as pd

# Lire les résultats
results_file = "baselines/ForecastCF/results/forecastcf_etth1_itransformer.csv"

if os.path.exists(results_file):
    df = pd.read_csv(results_file)
    print(f"Résultats ForecastCF ({len(df)} runs):\n")
    print(df)
    
    print("\n" + "="*80)
    print("Moyennes:")
    print("="*80)
    for col in ['validity_ratio', 'proximity', 'compactness', 'step_validity_auc']:
        if col in df.columns:
            mean = df[col].mean()
            std = df[col].std()
            print(f"{col:20s}: {mean:.4f} ± {std:.4f}")
else:
    print(f"✗ Fichier non trouvé: {results_file}")

## 5. Télécharger les résultats

In [ ]:
from google.colab import files

# Télécharger le fichier de résultats
if os.path.exists(results_file):
    files.download(results_file)
    print(f"✓ Fichier téléchargé: {results_file}")
else:
    print(f"✗ Fichier non trouvé: {results_file}")

## 6. (Optionnel) Comparer avec vos résultats RL

Si vous avez uploadé vos résultats RL, vous pouvez les comparer ici.

In [ ]:
# Uploader vos résultats RL
from google.colab import files
uploaded = files.upload()

# Le fichier uploadé sera dans le répertoire courant
rl_results_file = list(uploaded.keys())[0]
print(f"Fichier RL uploadé: {rl_results_file}")

In [ ]:
# Comparer
!python baselines/compare_results.py \
    --rl-results {rl_results_file} \
    --fcf-results {results_file} \
    --output-dir baselines/comparison_plots

In [ ]:
# Afficher les graphiques
from IPython.display import Image, display
import glob

plots = glob.glob("baselines/comparison_plots/*.png")
for plot in plots:
    print(f"\n{plot}")
    display(Image(plot))